In [28]:
import numpy as np
import pyspark.sql as spark
import pyspark.sql.functions as F
import faiss

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.clustering import KMeans
from pyspark import StorageLevel

from collections import defaultdict
from typing import (
    Union,
    List,
    Literal,
    Optional
)
class FaissSpark:
    """
    Реализация faiss на pyspark:
    ---------
        * **Большие данные** (> 1_000_000):
            1. Кластеризация данных;
            2. Используем faiss для кластера, а не для всего датасета.
        * **Маленькие данные**:
            1. Преобрауем датасет в pandas;
            2. Используем faiss на нашем датасете.
    """
    PERSIST_POLITIC = StorageLevel.MEMORY_AND_DISK

    def __init__(self,
                 n_neighbors: int = 1,
                 k: int = 1000,
                 seed: Union[int, None] = None,
                 feature_cols: Union[List[str], None] = None,
                 faiss_mode: Literal["base", "fast", "auto"] = "auto",
                 ):
        self.n_neighbors = n_neighbors
        self.k = k  # Количество кластеров
        # self.n_clusters = k  # ✅ Инициализируем n_clusters
        self.seed = seed
        self.feature_cols = feature_cols
        self.faiss_mode = faiss_mode

        self._kmeans_model = None
        self._index = None
        self._centroid_index = None
        self._centroid_list: List[np.ndarray] | None = None
        self._clustered_data: spark.DataFrame | None = None
        self._mode: Optional[Literal["base", "fast"]] = None

    def _vectorize_data(self, data: spark.DataFrame) -> spark.DataFrame:
        """
        Подготовка входных данных: векторизация и проверка на категориальные фичи.
        Все незакодированные фичи будут вызывать ошибку работы / (в дальнейшем просто выкидываться?).

        Parameters
        ----------
            data : SparkDataFrame
                Входные данные. Должен содержать:
                    - Числовые фичи;
                    - Заэнкоженные категориальные фичи;
        
        Returns
        -------
            vectorized_data: SparkDataFrame
                входной датасет с колонкой векторов из фичей.
        """
        if self.feature_cols is None:
            self.feature_cols = data.columns
        if len(set(map(lambda x: x[1], data.dtypes)).intersection(['varchar', 'string'])) > 0:
            raise TypeError("Unencoded categorical features are not allowed!")

        vecAssembler = VectorAssembler(inputCols=self.feature_cols,
                                     outputCol="features",
                                     handleInvalid="keep")
        
        return (
                    vecAssembler
                    .transform(data)
                    # .select("features")
                )

    def _direct_fit(self, data: spark.DataFrame) -> None:
        """
        Прямое вычисление faiss с выгрузкой данных на драйвер.
        """
        prepeared_data = self._vectorize_data(data)
        select_cols = ["features"]
        rows = prepeared_data.select(*select_cols).collect()
        X = np.array([list(row.features) for row in rows], dtype=np.float32)
        self._index = faiss.IndexFlatL2(X.shape[1])
        self._index.add(X)

    def _clustered_fit(self, data: spark.DataFrame) -> None:
        """
        Вычисление faiss с предварительной кластеризацией
        """
        prepeared_data = self._vectorize_data(data)
        df_clustered = self._clustering(prepeared_data)
        select_cols = ["cluster_id", "features"]
        self._clustered_data = (
            df_clustered
            .select(*select_cols)
            .persist(self.PERSIST_POLITIC)
        )
        self._clustered_data.count()
        
        X = np.array(self._centroid_list, dtype=np.float32)
        self._centroid_index= faiss.IndexFlatL2(X.shape[1])
        self._centroid_index.add(X)

    def _direct_predict(self, test_data: spark.DataFrame) -> List[List[Union[int, float]]]:
        rows = test_data.select("features").collect()
        X = np.array([list(row.features) for row in rows], dtype=np.float32)
        dist, indexes = self._index.search(X, k=self.n_neighbors)
        
        # Возвращаем список кортежей: [(query_idx, neighbor_idx, distance), ...]
        result = []
        for query_idx in range(len(X)):
            for i in range(self.n_neighbors):
                neighbor_idx = int(indexes[query_idx][i])
                distance = float(dist[query_idx][i])
                result.append((query_idx, neighbor_idx, distance))
        
        return result
    
    def _clustering_predict(self, test_data: spark.DataFrame) -> List[List[Union[int, float]]]:
        result = []
        rows = test_data.select("features").collect()
        X = np.array([list(row.features) for row in rows], dtype=np.float32)
        _, cluster_ids = self._centroid_index.search(X, k=1)
        # В spark KMeans центроиды возвращаются в соответствии с порядковым номером их кластера
        # Поэтому индекс сразу указывает нам на то, какой кластер нам нужен
        cluster_to_queries = defaultdict(list)
        for query_idx, cluster_idx in enumerate(cluster_ids.flatten()):
            cluster_to_queries[cluster_idx].append(query_idx)

        for cluster_idx, query_idx_list in cluster_to_queries.items():
            cluster_rows = (
                self._clustered_data
                .filter(F.col('cluster_id') == cluster_idx)
                .select("features")
                .collect()
            )
            cluster_data = np.array(
                [list(row.features) for row in cluster_rows], 
                dtype=np.float32
            )
            tmp_index = faiss.IndexFlatL2(cluster_data.shape[1])
            tmp_index.add(cluster_data)
            k = min(self.n_neighbors, len(cluster_data))
            for query_idx in query_idx_list:
                r_dist, r_indexes = tmp_index.search(X[query_idx: query_idx+1], k=k)
                # Возвращаем результат: (query_idx, neighbor_idx, distance)
                for i in range(self.n_neighbors):
                    neighbor_idx = int(r_indexes[0][i])
                    distance = float(r_dist[0][i])
                    result.append((query_idx, neighbor_idx, distance))
        # for query_idx in range(len(X)):
        #     query_vector = X[query_idx:query_idx+1]
        #     cluster_id = int(cluster_ids[query_idx][0])
            
            # cluster_rows = (
            #     self._clustered_data
            #     .filter(F.col('cluster_id') == cluster_idx)
            #     .select("features")
            #     .collect()
            # )
            
        #     # Проверка на пустой кластер
        #     if len(cluster_rows) == 0:
        #         result.append((query_idx, -1, float('inf')))
        #         continue
            
        #     cluster_data = np.array(
        #         [list(row.features) for row in cluster_rows], 
        #         dtype=np.float32
        #     )
            
        #     tmp_index = faiss.IndexFlatL2(cluster_data.shape[1])
        #     tmp_index.add(cluster_data)
            
        #     k = min(self.n_neighbors, len(cluster_data))
        #     r_dist, r_indexes = tmp_index.search(query_vector, k=k)
            
        #     # Возвращаем результат: (query_idx, neighbor_idx, distance)
        #     for i in range(self.n_neighbors):
        #         neighbor_idx = int(r_indexes[0][i])
        #         distance = float(r_dist[0][i])
        #         result.append((query_idx, neighbor_idx, distance))
        
        return result
        

    def _clustering(self, data: spark.DataFrame) -> spark.DataFrame:
        """
        Разбиение на кластеры для дальнейшего использования faiss для каждого кластера.

        Parametrs
        ----------
            data : SparkDataFrame
                Подготовленные данные с помощью *_vectorize_data*
        
        Returns
        -------
            df_clustered : SparkDataFrame
                Датафрейм с колонкой пренадлежности к кластеру.
        """
        # vectorized_data = self._vectorize_data(data)
        self._kmeans_model = KMeans(k=self.k, 
                                    seed=self.seed,
                                    featuresCol="features",
                                    predictionCol='cluster_id')
        self._kmeans_model = self._kmeans_model.fit(data)
        df_clustered = self._kmeans_model.transform(data)
        self._centroid_list = self._kmeans_model.clusterCenters()

        return df_clustered
    
    def _calculation_mode(self, count: int):
        """
        Выбор типа вычисления faiss.
        """
        if self.faiss_mode == "base":
            return "base"
        if self.faiss_mode == "fast":
            return "fast"
        if self.faiss_mode == "auto":
            if count > 1_000_000:
                return "fast"
            return "base"
    
    def fit(self, data: spark.DataFrame) -> "FaissSpark":
        """
        Fit.
        """
        count = data.count()
        self._mode = self._calculation_mode(count)

        if self._mode == "base":
            self._direct_fit(data)
        if self._mode == "fast":
            self._clustered_fit(data)

        return self
    
    def predict(self, test_data: spark.DataFrame):
        """
        Predict.
        
        Returns
        -------
        result : List[Tuple[int, int, float]]
            Список кортежей: (порядковый номер записи в test_data, найденный близнец, расстояние до него)
        """
        prepeared_data = self._vectorize_data(test_data)
        if self._mode == "base":
            result = self._direct_predict(prepeared_data)
        elif self._mode == "fast":
            result = self._clustering_predict(prepeared_data)
        else:
            raise RuntimeError("Модель не обучена. Вызовите fit() перед predict().")
        
        return result

    def unpersist(self):
        """
        Подчистить ресурсы spark.
        """
        if self._clustered_data is not None:
            self._clustered_data.unpersist()
            self._clustered_data = None
        
    def __del__(self):
        self.unpersist()
            

In [2]:
from pyspark.sql import SparkSession, functions as F

In [3]:
session = (
            SparkSession.builder
            .master("local[*]")
            .config("spark.driver.memory", "4g")
            .config("spark.executo|r.memory", "4g")
            .config("spark.memory.fraction", "0.8") 
            .config("spark.memory.storageFraction", "0.3") 
            .getOrCreate()
          )

26/03/06 12:56:10 WARN Utils: Your hostname, eric-Katana-17-B12UCR resolves to a loopback address: 127.0.1.1; using 10.240.72.74 instead (on interface wlo1)
26/03/06 12:56:10 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/06 12:56:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
data = (
            session.read.format('csv')
            .option("header", "true")
            .option("inferSchema", "true")
            .option("ignoreLeadingWhiteSpace", "true") 
            .option("ignoreTrailingWhiteSpace", "true")
            .load("../datasets/data.csv")
        )
data = data.drop("_c0")
data.show()

+-------+------------+-----+----------+------------------+----+------+----------+
|user_id|signup_month|treat|pre_spends|       post_spends| age|gender|  industry|
+-------+------------+-----+----------+------------------+----+------+----------+
|    0.0|         0.0|  0.0|     519.5| 424.6666666666667|39.0|     M| Logistics|
|    1.0|         0.0|  0.0|     500.5| 423.1111111111111|19.0|     F| Logistics|
|    2.0|         0.0|  0.0|     499.0|423.44444444444446|29.0|     M|E-commerce|
|    3.0|         2.0|  1.0|     494.0| 530.5555555555555|38.0|     M|E-commerce|
|    4.0|         9.0|  1.0|     455.0| 451.8888888888889|53.0|     F| Logistics|
|    5.0|         1.0|  1.0|     536.0| 518.3333333333334|22.0|     F| Logistics|
|    6.0|         4.0|  1.0|     493.5|500.77777777777777|21.0|     M|E-commerce|
|    7.0|         8.0|  1.0|     484.0| 447.3333333333333|69.0|     F|E-commerce|
|    8.0|         0.0|  0.0|     473.5|413.22222222222223|34.0|     M|E-commerce|
|    9.0|       

In [6]:
prepeared_data = data.select(*[
    "user_id",
    "signup_month",
    "treat",
    "pre_spends",
    "post_spends",
    "age"
    # "*"
]).dropna()
prepeared_data.show()

+-------+------------+-----+----------+------------------+----+
|user_id|signup_month|treat|pre_spends|       post_spends| age|
+-------+------------+-----+----------+------------------+----+
|    0.0|         0.0|  0.0|     519.5| 424.6666666666667|39.0|
|    1.0|         0.0|  0.0|     500.5| 423.1111111111111|19.0|
|    2.0|         0.0|  0.0|     499.0|423.44444444444446|29.0|
|    3.0|         2.0|  1.0|     494.0| 530.5555555555555|38.0|
|    4.0|         9.0|  1.0|     455.0| 451.8888888888889|53.0|
|    5.0|         1.0|  1.0|     536.0| 518.3333333333334|22.0|
|    6.0|         4.0|  1.0|     493.5|500.77777777777777|21.0|
|    7.0|         8.0|  1.0|     484.0| 447.3333333333333|69.0|
|    8.0|         0.0|  0.0|     473.5|413.22222222222223|34.0|
|    9.0|         2.0|  1.0|     513.5| 518.5555555555555|39.0|
|   11.0|         0.0|  0.0|     493.5|409.77777777777777|25.0|
|   12.0|        10.0|  1.0|     474.0|442.22222222222223|51.0|
|   13.0|         3.0|  1.0|     503.5| 

In [7]:
prepeared_data.count()

9001

In [13]:
weights = [0.99, 0.01]
random_state = 21

In [34]:
train_data, test_data = prepeared_data.randomSplit(weights=weights)

In [35]:
print(f"Size of train data: {train_data.count()}")
print(f"Size of test data: {test_data.count()}")

Size of train data: 8906
Size of test data: 95


In [29]:
fast_matching = FaissSpark(
    n_neighbors=3,
    k=100,
    faiss_mode='fast'
)

direct_matching = FaissSpark(
    n_neighbors=3,
    k=100,
    faiss_mode='base'
)

In [30]:
fast_matching.fit(train_data)
print("Cluster fit done")
direct_matching.fit(train_data)
print("Direct fit done")

Cluster fit done
Direct fit done


In [31]:
fast_result = fast_matching.predict(test_data)
direct_result = direct_matching.predict(test_data)

In [32]:
fast_result

[(0, 29, 436.74371337890625),
 (0, 15, 446.901123046875),
 (0, 5, 572.2249145507812),
 (1, 17, 245.02792358398438),
 (1, 25, 313.7784423828125),
 (1, 23, 548.975830078125),
 (2, 37, 47.02780532836914),
 (2, 32, 241.41949462890625),
 (2, 30, 363.0),
 (3, 6, 576.595458984375),
 (3, 2, 886.0027465820312),
 (3, 10, 887.6327514648438),
 (4, 16, 150.44430541992188),
 (4, 20, 246.02761840820312),
 (4, 7, 276.45672607421875),
 (5, 107, 424.4483337402344),
 (5, 92, 1077.3619384765625),
 (5, 94, 1150.9293212890625),
 (6, 59, 686.4441528320312),
 (6, 60, 804.5948486328125),
 (6, 58, 860.77734375),
 (7, 29, 234.6048126220703),
 (7, 26, 284.75262451171875),
 (7, 28, 296.0396728515625),
 (8, 49, 240.22491455078125),
 (8, 48, 260.52154541015625),
 (8, 42, 295.7779846191406),
 (9, 18, 1430.694091796875),
 (9, 11, 1487.447265625),
 (9, 35, 1612.743896484375),
 (10, 92, 1881.197509765625),
 (10, 99, 1939.2623291015625),
 (10, 53, 2457.455810546875),
 (11, 113, 227.69447326660156),
 (11, 114, 251.4103546

In [33]:
direct_result

[(0, 210, 436.75),
 (0, 196, 446.9375),
 (0, 185, 572.1875),
 (1, 258, 245.0),
 (1, 266, 313.75),
 (1, 264, 548.875),
 (2, 469, 47.0),
 (2, 464, 241.375),
 (2, 461, 363.0),
 (3, 585, 576.625),
 (3, 577, 886.0),
 (3, 593, 887.75),
 (4, 767, 150.5),
 (4, 771, 246.0),
 (4, 758, 276.5),
 (5, 859, 424.5),
 (5, 843, 1077.25),
 (5, 845, 1150.75),
 (6, 920, 686.5),
 (6, 921, 804.75),
 (6, 919, 860.75),
 (7, 1128, 234.5),
 (7, 1125, 284.75),
 (7, 1127, 296.0),
 (8, 1148, 240.25),
 (8, 1147, 260.5),
 (8, 1141, 296.0),
 (9, 1235, 1430.5),
 (9, 1203, 1453.0),
 (9, 1228, 1487.5),
 (10, 1309, 1881.5),
 (10, 1316, 1939.0),
 (10, 1270, 2457.5),
 (11, 1337, 164.0),
 (11, 1331, 228.0),
 (11, 1332, 251.5),
 (12, 1439, 199.0),
 (12, 1421, 279.5),
 (12, 1427, 455.0),
 (13, 1432, 117.5),
 (13, 1445, 362.0),
 (13, 1455, 1163.5),
 (14, 1463, 221.5),
 (14, 1476, 427.0),
 (14, 1464, 432.5),
 (15, 1480, 52.0),
 (15, 1479, 146.5),
 (15, 1478, 219.5),
 (16, 1862, 1585.0),
 (16, 1889, 1845.0),
 (16, 1895, 1943.0),


In [36]:
session

In [38]:
session.sparkContext.applicationId

'local-1772790972150'

In [42]:
session.stop()

In [16]:
import numpy as np

d = 64                           # dimension
nb = 100000                      # database size
nq = 10000                       # nb of queries
np.random.seed(1234)             # make reproducible
xb = np.random.random((nb, d)).astype('float32')
xb[:, 0] += np.arange(nb) / 1000.
xq = np.random.random((nq, d)).astype('float32')
xq[:, 0] += np.arange(nq) / 1000.

import faiss                   # make faiss available
index = faiss.IndexFlatL2(d)   # build the index
print(index.is_trained)
index.add(xb)                  # add vectors to the index
print(index.ntotal)

k = 4                          # we want to see 4 nearest neighbors
D, I = index.search(xb[:5], k) # sanity check
print(I)
print(D)
D, I = index.search(xq, k)     # actual search
print(I[:5])                   # neighbors of the 5 first queries
print(I[-5:])                  # neighbors of the 5 last queries

True
100000
[[  0 393 363  78]
 [  1 555 277 364]
 [  2 304 101  13]
 [  3 173  18 182]
 [  4 288 370 531]]
[[0.        7.1751738 7.20763   7.2511625]
 [0.        6.3235645 6.684581  6.799946 ]
 [0.        5.7964087 6.391736  7.2815123]
 [0.        7.2779055 7.5279875 7.662846 ]
 [0.        6.7638035 7.2951202 7.3688145]]
[[ 381  207  210  477]
 [ 526  911  142   72]
 [ 838  527 1290  425]
 [ 196  184  164  359]
 [ 526  377  120  425]]
[[ 9900 10500  9309  9831]
 [11055 10895 10812 11321]
 [11353 11103 10164  9787]
 [10571 10664 10632  9638]
 [ 9628  9554 10036  9582]]


In [ ]:
index

TypeError: 'IndexFlatL2' object is not subscriptable